# Scholarship Eligibility Prediction
### Course: AD4V71 / AD5302 – Data Science
**Team Members:**
- **Kirutick Siddhesh V** (Regd No: 210425243120, Section C)
- **Krishna H** (Regd No: 210425243124, Section C)

---

## 1. Project Overview & Problem Statement

### The Problem
Higher education scholarship distribution is traditionally a labor-intensive, multi-tiered administrative process. Institutional review committees must manually cross-reference hundreds of student applications against complex governmental and institutional eligibility criteria (academic cutoffs, income limits, affirmative reservation categories, first-generation status). This manual workflow is:
1. **Time-consuming**: Causes prolonged delays in scholarship sanction and student disbursement.
2. **Subject to Inconsistency**: Human reviewer bias or varying subjective interpretations across departments.
3. **Resource Strained**: Administrative burdens distract academic staff from core educational duties.

### The Objective
To design and benchmark an end-to-end Machine Learning classification system that accurately screens and predicts student scholarship eligibility based on multidimensional academic, socioeconomic, and demographic features.

### Scope & System Boundaries
- **IN SCOPE**: Data collection, preprocessing, exploratory data analysis, feature engineering, classification model training (Logistic Regression, Decision Tree, Random Forest, Naïve Bayes), performance evaluation, and decision-support prediction.
- **OUT OF SCOPE**: This system **does not** disburse funds, verify forged certificates, or make official government legal determinations. It functions strictly as an intelligent screening and decision-support tool.


## 2. Imports and Environment Setup
We load standard data science libraries:
- `pandas` & `numpy` for data manipulation and vectorized calculations
- `matplotlib` & `seaborn` for visual analytics
- `scikit-learn` for preprocessing pipelines, cross-validation, and classification models
- `joblib` for model serialization


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import display

# Ensure project root is accessible
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.autolayout": True
})

# Reproducibility seed
SEED = 42
np.random.seed(SEED)
print("Libraries imported successfully. Environment initialized.")


## 3. Dataset Loading
We load the student scholarship dataset (`data/raw/scholarship_data.csv`).
This is a **synthetic reference dataset** (1,000 records) created specifically for academic machine learning experimentation, using general scholarship categories and AISHE demographic parameters as structural inspiration.


In [ ]:
data_path = os.path.join("..", "data", "raw", "scholarship_data.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "raw", "scholarship_data.csv")

df = pd.read_csv(data_path)
print(f"Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head(10)


## 4. Dataset Inspection
We systematically inspect:
- Data types of each column
- Non-null counts and missing values
- Duplicate observations
- High-level descriptive statistics


In [ ]:
print("--- DATASET INFO ---")
df.info()

print("\n--- MISSING VALUES PER FEATURE ---")
print(df.isnull().sum())

print(f"\n--- DUPLICATE ROWS: {df.duplicated().sum()} ---")

print("\n--- DESCRIPTIVE STATISTICS (NUMERICAL) ---")
display(df.describe().T)


## 5. Data Preprocessing
### Cleaning Steps:
1. **Deduplication**: Verify that no identical student records exist.
2. **Missing Value Imputation**: Median for numerical variables, mode for nominal categorical variables.
3. **Identifier Isolation**: Column `StudentID` is dropped from model features because unique applicant IDs have no generalizable predictive value and would otherwise cause memorization/leakage.


In [ ]:
# Remove duplicates if any
initial_count = len(df)
df = df.drop_duplicates()
print(f"Deduplication complete. Retained {len(df)} of {initial_count} rows.")

# Separate features and target
X = df.drop(columns=['StudentID', 'Eligibility'])
y = df['Eligibility']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")


## 6. Exploratory Data Analysis (EDA)
We visually examine key univariate and bivariate distributions across academic and financial attributes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
palette = {"Eligible": "#2b8a3e", "Not Eligible": "#e03131"}

# 1. Target Distribution
counts = df['Eligibility'].value_counts()
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, ax=axes[0], palette=["#2b8a3e", "#c92a2a"], legend=False)
axes[0].set_title("Scholarship Eligibility Count")
axes[0].set_xlabel("Eligibility Status")
axes[0].set_ylabel("Number of Students")
for i, c in enumerate(counts.values):
    axes[0].text(i, c + 15, f"{c} ({c/len(df)*100:.1f}%)", ha="center", fontweight="bold")

axes[1].pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=140,
           colors=["#2b8a3e", "#c92a2a"], wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2))
axes[1].set_title("Class Proportion (Moderate Imbalance)")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2. Family Income Distribution
sns.histplot(data=df, x='FamilyIncome', hue='Eligibility', kde=True, ax=axes[0], palette=palette, bins=25)
axes[0].set_title("Family Income Distribution (KDE & Histogram)")
axes[0].set_xlabel("Annual Household Income (INR)")

sns.boxplot(data=df, x='Eligibility', y='FamilyIncome', hue='Eligibility', ax=axes[1], palette=palette, legend=False)
axes[1].set_title("Family Income by Eligibility (Boxplot)")
axes[1].set_ylabel("Annual Household Income (INR)")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 3. 12th Board Marks Distribution
sns.histplot(data=df, x='12thMarks', hue='Eligibility', kde=True, ax=axes[0], palette=palette, bins=25)
axes[0].set_title("12th Marks Distribution (KDE & Histogram)")
axes[0].set_xlabel("12th Board Marks (%)")

sns.boxplot(data=df, x='Eligibility', y='12thMarks', hue='Eligibility', ax=axes[1], palette=palette, legend=False)
axes[1].set_title("12th Marks by Eligibility (Boxplot)")
axes[1].set_ylabel("12th Board Marks (%)")
plt.show()


In [ ]:
plt.figure(figsize=(10, 5))
comm_order = ['SC', 'ST', 'MBC', 'BC', 'OC']
sns.countplot(data=df, x='Community', hue='Eligibility', order=comm_order, palette=palette)
plt.title("Scholarship Eligibility Count across Communities", pad=12)
plt.xlabel("Community Reservation Category")
plt.ylabel("Student Count")
plt.legend(title="Eligibility")
plt.show()


## 7. Correlation Analysis
### Understanding Pearson's Correlation Coefficient ($r$)
Pearson's $r$ measures linear association between two continuous variables on a scale from $-1.0$ (perfect inverse relationship) to $+1.0$ (perfect direct relationship):
$$r = \frac{\sum (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum (X_i - \bar{X})^2 \sum (Y_i - \bar{Y})^2}}$$

### Encoding Note:
We map **Eligible = 1** and **Not Eligible = 0**.
- **Negative correlation with FamilyIncome ($r \approx -0.58$)**: As annual income rises, the probability of qualifying for financial assistance falls significantly.
- **Positive correlation with 12thMarks ($r \approx 0.30$)**: Higher academic performance increases the probability of qualifying for merit-cum-means awards.
- **Correlation $\neq$ Causation**: High marks alone do not cause approval without satisfying affirmative means-tested criteria.


In [ ]:
df_corr = df.copy()
df_corr['Target_Numeric'] = (df_corr['Eligibility'] == 'Eligible').astype(int)

num_cols = ['FamilyIncome', '12thMarks', 'Target_Numeric']
corr_matrix = df_corr[num_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("Correlation Matrix: Financial & Academic Features vs Eligibility")
plt.show()

r_income = corr_matrix.loc['FamilyIncome', 'Target_Numeric']
r_marks = corr_matrix.loc['12thMarks', 'Target_Numeric']

print(f"Measured r(FamilyIncome, Target): {r_income:.2f} (PPT Reference: -0.58)")
print(f"Measured r(12thMarks, Target):    {r_marks:.2f} (PPT Reference: 0.30)")


## 8. Feature Engineering Pipeline & Preventing Data Leakage
To ensure strict statistical validity:
1. **No Data Leakage**: Transformations must be fitted **strictly on the training partition** and then applied to the test partition.
2. **StandardScaler**: Applied to continuous numerical columns (`FamilyIncome`, `12thMarks`) to normalize scale variance $\mu=0, \sigma=1$.
3. **OneHotEncoder**: Applied to discrete nominal features (`Community`, `FirstGraduate`, `District`, `CollegeType`, `Course`, `Gender`) with `handle_unknown='ignore'`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

num_features = ['FamilyIncome', '12thMarks']
cat_features = ['Gender', 'Community', 'FirstGraduate', 'District', 'CollegeType', 'Course']

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ]
)
print("ColumnTransformer preprocessor initialized (with median/mode imputation & zero leakage).")


## 9. Stratified Train/Test Split
Because the dataset is moderately imbalanced (~63.8% Eligible vs ~36.2% Not Eligible), a naive random split risks drawing disproportionate class splits in the test set.
We enforce `stratify=y` to maintain exact class proportions in both sets:
- **80% Training set** (800 students)
- **20% Test set** (200 students)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print(f"Training split: {X_train.shape[0]} samples")
print(f"Testing split:  {X_test.shape[0]} samples")
print(f"Train class balance:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"Test class balance:\n{y_test.value_counts(normalize=True).round(3)}")


## 10. Model Training (Four Classifiers)
We train and compare four fundamental classification algorithms:
1. **Logistic Regression**: Linear decision boundary using sigmoid log-odds.
2. **Decision Tree**: Rule-based partition splitting on Gini impurity.
3. **Random Forest**: Ensemble of bagged decision trees with feature sub-sampling to lower variance.
4. **Naïve Bayes (Gaussian)**: Probabilistic classifier applying Bayes' theorem with feature conditional independence.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=SEED),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED),
    "Naïve Bayes": GaussianNB()
}

fitted_pipelines = {}
for name, clf in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    fitted_pipelines[name] = pipe
    print(f"Trained: {name}")


## 11. Model Evaluation & Performance Comparison Table
### Why F1-Score Matters:
In an imbalanced classification setting (63.8% majority class), a dummy model predicting 'Eligible' every time would achieve 63.8% accuracy while being completely useless.
- **Precision**: $\frac{TP}{TP + FP}$ (Avoids awarding scholarships to ineligible candidates)
- **Recall**: $\frac{TP}{TP + FN}$ (Avoids rejecting genuinely eligible, needy students)
- **F1-Score**: Harmonic mean of Precision and Recall:
$$F1 = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []
for name, pipe in fitted_pipelines.items():
    y_pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, pos_label="Eligible"), 4),
        "Recall": round(recall_score(y_test, y_pred, pos_label="Eligible"), 4),
        "F1-Score": round(f1_score(y_test, y_pred, pos_label="Eligible"), 4)
    })

comparison_df = pd.DataFrame(results)
display(comparison_df)


In [ ]:
melted_df = comparison_df.melt(
    id_vars=["Model"],
    value_vars=["Accuracy", "Precision", "Recall", "F1-Score"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(10, 5))
sns.barplot(data=melted_df, x="Model", y="Score", hue="Metric", palette=["#1971c2", "#2f9e44", "#f59f00", "#e03131"])
plt.title("Model Comparison across Evaluation Metrics", pad=12)
plt.ylim(0.70, 1.02)
plt.legend(title="Metric", loc="lower right")
plt.show()


## 12. Confusion Matrix Diagnostics
The confusion matrix tracks exact outcomes:
- **True Negative (TN)**: Ineligible correctly predicted Ineligible.
- **False Positive (FP)**: Ineligible mistakenly predicted Eligible (Misallocation risk).
- **False Negative (FN)**: Eligible student mistakenly denied (Hardship risk).
- **True Positive (TP)**: Eligible student correctly identified.


In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.flatten()
labels = ["Not Eligible", "Eligible"]

for idx, (name, pipe) in enumerate(fitted_pipelines.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx], xticklabels=labels, yticklabels=labels, cbar=False)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, pos_label="Eligible")
    axes[idx].set_title(f"{name}\n(Accuracy: {acc*100:.1f}%, F1: {f1:.3f})")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("True")

plt.tight_layout()
plt.show()


## 13. Random Forest Feature Importance
Random Forest computes the mean Gini Impurity reduction contributed by each feature across all 100 constituent decision trees.
**Important Note:** Feature importance indicates how heavily the decision trees relied on each feature to partition this dataset. It reflects **predictive contribution within the model**, NOT real-world causality.


In [ ]:
rf_pipe = fitted_pipelines["Random Forest"]
preproc = rf_pipe.named_steps['preprocessor']
rf_clf = rf_pipe.named_steps['classifier']

# Extract transformed feature names
cat_step = preproc.named_transformers_['cat']
cat_encoder = cat_step.named_steps['encoder'] if hasattr(cat_step, 'named_steps') else cat_step
cat_names = list(cat_encoder.get_feature_names_out(cat_features))
all_feature_names = num_features + cat_names

importances = rf_clf.feature_importances_
fi_df = pd.DataFrame({"Feature": all_feature_names, "Importance": importances}).sort_values(by="Importance", ascending=False)

top_10 = fi_df.head(10).sort_values(by="Importance", ascending=True)

plt.figure(figsize=(9, 5))
colors = ["#1971c2" if f in ["FamilyIncome", "12thMarks"] else "#748ffc" for f in top_10["Feature"]]
plt.barh(top_10["Feature"], top_10["Importance"], color=colors)
plt.title("Random Forest Top 10 Feature Importances", pad=12)
plt.xlabel("Mean Impurity Reduction (Importance)")
for idx, val in enumerate(top_10["Importance"]):
    plt.text(val + 0.005, idx, f"{val:.3f}", va="center", fontsize=9)
plt.show()

print("Top 3 Measured Predictors:")
for i, row in fi_df.head(3).iterrows():
    print(f"  {row['Feature']}: {row['Importance']*100:.1f}%")


## 14. Model Selection & Example Predictions
We select **Random Forest** based on superior F1-Score and robust generalization across non-linear policy criteria.
We now demonstrate test predictions across several real-world applicant profiles.


In [ ]:
# Export the best model pipeline
best_model_path = os.path.join("..", "models", "best_model.joblib")
if not os.path.exists(os.path.dirname(best_model_path)):
    best_model_path = os.path.join("models", "best_model.joblib")

joblib.dump(fitted_pipelines["Random Forest"], best_model_path)
print(f"Serialized best model pipeline to: {best_model_path}")


In [ ]:
# Example inference test cases
test_cases = [
    {
        "Candidate": "Profile A (SC, Low Income, High Merit)",
        "Gender": "Female", "Community": "SC", "FamilyIncome": 85000,
        "12thMarks": 86.0, "FirstGraduate": "Yes", "District": "Madurai",
        "CollegeType": "Government", "Course": "Engineering"
    },
    {
        "Candidate": "Profile B (OC, High Income, Moderate Merit)",
        "Gender": "Male", "Community": "OC", "FamilyIncome": 520000,
        "12thMarks": 64.0, "FirstGraduate": "No", "District": "Chennai",
        "CollegeType": "Private", "Course": "Management"
    },
    {
        "Candidate": "Profile C (MBC, Moderate Income, First Graduate)",
        "Gender": "Female", "Community": "MBC", "FamilyIncome": 135000,
        "12thMarks": 74.0, "FirstGraduate": "Yes", "District": "Salem",
        "CollegeType": "Government Aided", "Course": "Arts & Science"
    }
]

df_cases = pd.DataFrame(test_cases)
features_only = df_cases.drop(columns=["Candidate"])

probs = rf_pipe.predict_proba(features_only)
preds = rf_pipe.predict(features_only)
eligible_col = list(rf_pipe.classes_).index("Eligible")

df_cases["Predicted Status"] = preds
df_cases["Eligible Probability (%)"] = (probs[:, eligible_col] * 100).round(1)

display(df_cases[["Candidate", "Community", "FamilyIncome", "12thMarks", "Predicted Status", "Eligible Probability (%)"]])


## 15. Conclusion & Presentation Consistency Review

### Key Findings:
1. **Performance**: Random Forest achieved **93.0% Accuracy** and **0.944 F1-Score**, outperforming Decision Tree (90.5%), Logistic Regression (86.0%), and Naïve Bayes (84.0%).
2. **Predictor Ranking**: Confirmed that `FamilyIncome` (46.9% model feature importance) and `12thMarks` (21.3% model feature importance) are the top predictors in the model.
3. **Tree-Based Ensembles**: Random Forest handles multi-variable non-linear threshold rules effectively through bootstrap aggregation and feature sub-sampling.

### Alignment with Project Presentation:
- **Presentation Reference (Slide 10)**: Random Forest Accuracy = 95.5%, F1-Score = 0.966.
- **Measured Implementation**: Random Forest Accuracy = 93.0%, F1-Score = 0.944.
- **Recommended PPT Update**: Update presentation to state: *"Random Forest achieved 93.0% accuracy and 0.944 F1-score on the held-out test set."*
- **Label Variation Note**: Random label variation was introduced during synthetic dataset generation to avoid creating a perfectly deterministic target. The model ranking and feature importance hierarchy remain 100% consistent with the presentation narrative.

### Project Limitations:
- Dataset is a 1,000-record synthetic reference dataset; real-world deployment requires authentic administrative data.
- Feature importance measures predictive association within the model, not real-world causality.
- Predictions represent academic pattern recognition, not official scholarship sanction.
